### In this notebook:
## Generate separate crime buffer, crime density, and light density safety scores

#### Preparation:


In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import geopandas as gpd
from shapely.geometry import box, Point
import osmnx as ox
from sklearn.neighbors import KernelDensity
from shapely import wkt
from shapely.geometry import LineString, MultiLineString
import math
import os

EARTH_RADIUS_M = 6_371_000.0
density_sigma = 40
light_alpha = 0.2
cross_beta = 0.4



In [ ]:

edges_df = pd.read_csv('../crime/data/edges/edges_seattle.csv')

if edges_df["geometry"].dtype == object:
    edges_df["geometry"] = edges_df["geometry"].apply(wkt.loads)

%run ../crime/crime_density_edge_mapping.ipynb
%run ../Lightpoles/lightpoles_condensed.ipynb
%run ../crime/run_all.ipynb

In [10]:
scores_df = cbs_to_df(edges_df)
print(scores_df.describe())

           edge_id             u             v           key        length  \
count  25886.00000  2.588600e+04  2.588600e+04  25886.000000  25886.000000   
mean   12942.50000  8.282471e+09  8.282471e+09      0.009349     27.697297   
std     7472.78887  2.988861e+09  2.988861e+09      0.103211     29.923901   
min        0.00000  2.993814e+07  2.993814e+07      0.000000      0.239832   
25%     6471.25000  5.767399e+09  5.767399e+09      0.000000      7.884306   
50%    12942.50000  8.920207e+09  8.920207e+09      0.000000     17.274127   
75%    19413.75000  1.075794e+10  1.075794e+10      0.000000     37.951390   
max    25885.00000  1.299323e+10  1.299323e+10      3.000000    523.929678   

         buffer_day  buffer_night  buffer_blended_day  buffer_blended_night  
count  25886.000000  25886.000000        25886.000000          25886.000000  
mean       0.863901      0.829490            0.817711              0.797160  
std        0.277765      0.298467            0.300222          

In [11]:
scores_df.describe()

           edge_id             u             v           key        length  \
count  25886.00000  2.588600e+04  2.588600e+04  25886.000000  25886.000000   
mean   12942.50000  8.282471e+09  8.282471e+09      0.009349     27.697297   
std     7472.78887  2.988861e+09  2.988861e+09      0.103211     29.923901   
min        0.00000  2.993814e+07  2.993814e+07      0.000000      0.239832   
25%     6471.25000  5.767399e+09  5.767399e+09      0.000000      7.884306   
50%    12942.50000  8.920207e+09  8.920207e+09      0.000000     17.274127   
75%    19413.75000  1.075794e+10  1.075794e+10      0.000000     37.951390   
max    25885.00000  1.299323e+10  1.299323e+10      3.000000    523.929678   

         buffer_day  buffer_night  buffer_blended_day  buffer_blended_night  
count  25886.000000  25886.000000        25886.000000          25886.000000  
mean       0.863901      0.829490            0.817711              0.797160  
std        0.277765      0.298467            0.300222          

In [12]:

scores_df = cds_to_df(scores_df, density_sigma)

In [13]:
scores_df = ls_to_df(scores_df)


In [14]:
print(scores_df.describe())
scores_df.to_csv('data/safety_scores.csv')

           edge_id             u             v           key        length  \
count  25886.00000  2.588600e+04  2.588600e+04  25886.000000  25886.000000   
mean   12942.50000  8.282471e+09  8.282471e+09      0.009349     27.697297   
std     7472.78887  2.988861e+09  2.988861e+09      0.103211     29.923901   
min        0.00000  2.993814e+07  2.993814e+07      0.000000      0.239832   
25%     6471.25000  5.767399e+09  5.767399e+09      0.000000      7.884306   
50%    12942.50000  8.920207e+09  8.920207e+09      0.000000     17.274127   
75%    19413.75000  1.075794e+10  1.075794e+10      0.000000     37.951390   
max    25885.00000  1.299323e+10  1.299323e+10      3.000000    523.929678   

         buffer_day  buffer_night  buffer_blended_day  buffer_blended_night  \
count  25886.000000  25886.000000        25886.000000          25886.000000   
mean       0.863901      0.829490            0.817711              0.797160   
std        0.277765      0.298467            0.300222       

In [10]:
df = pd.read_csv("../routing/all_scored_edges_filtered_with_ai.csv")
cds = pd.read_csv("../crime_density/density_scores/cds_67.csv")
light = pd.read_csv("newlightpoles.csv")

/var/folders/92/3gykpnwj75zc04j2_wnxhtcw0000gn/T/ipykernel_93187/2087456110.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../routing/all_scored_edges_filtered_with_ai.csv")
/var/folders/92/3gykpnwj75zc04j2_wnxhtcw0000gn/T/ipykernel_93187/2087456110.py:2: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  cds = pd.read_csv("../crime_density/density_scores/cds_67.csv")


In [11]:
scores_df = df.copy()
# add light to night
scores_df['density_light'] = (cds['safety_score_night_density'] * (1+0.2 * light['ldensity'])).clip(0, 1)

In [13]:
# cross blend for day
scores_df["safety_score_day_final"] = (scores_df["safety_score_day_density"] - 0.4 * (1 - scores_df["density_light"])).clip(0, 1)
scores_df["safety_score_night_final"] = (scores_df["density_light"] - 0.4 * (1 - scores_df["safety_score_day_density"])).clip(0, 1)

In [16]:
scores_df["safety_score_day_final"] = scores_df["safety_score_day_final"] * 0.9 + 0.1
scores_df["safety_score_night_final"] = scores_df["safety_score_night_final"] * 0.9 + 0.1
df["safety_score_day_density"] = scores_df["safety_score_day_final"]
df["safety_score_night_density"] = scores_df["safety_score_night_final"]
df.to_csv("../routing/all_scored_edges_filtered_with_ai.csv")

In [ ]:
scores_df[f'{keywords[i]}_day_final_score'] = (
        scores_df[f'{keywords[i]}_day'] - cross_beta * (1 - scores_df[f'{keywords[i]}_night'])
    ).clip(0, 1)

    scores_df[f'{keywords[i]}_night_final_score'] = (
        scores_df[f'{keywords[i]}_night'] - cross_beta * (1 - scores_df[f'{keywords[i]}_day'])
    ).clip(0, 1)

In [ ]:
crime_night_cols = ['buffer_night', 'density_night']
keywords = ['buffer', 'density']
num_cols = len(crime_night_cols)
num_words = len(keywords)

for i in range(num_cols):
    scores_df[f'{keywords[i]}_light'] = (scores_df[crime_night_cols[i]] * (1+light_alpha * scores_df['light_score'])).clip(0, 1)



KeyError: 'density_night'

In [16]:
for i in range(num_words):
 
    scores_df[f'{keywords[i]}_day_final_score'] = (
        scores_df[f'{keywords[i]}_day'] - cross_beta * (1 - scores_df[f'{keywords[i]}_night'])
    ).clip(0, 1)

    scores_df[f'{keywords[i]}_night_final_score'] = (
        scores_df[f'{keywords[i]}_night'] - cross_beta * (1 - scores_df[f'{keywords[i]}_day'])
    ).clip(0, 1)

In [17]:
scores_df.to_csv('data/safety_scores.csv')

In [19]:
scores_df.columns

Index(['edge_id', 'u', 'v', 'key', 'osmid', 'highway', 'oneway', 'reversed',
       'length', 'lanes', 'maxspeed', 'name', 'access', 'service', 'geometry',
       'bridge', 'tunnel', 'ref', 'buffer_day', 'buffer_night',
       'buffer_blended_day', 'buffer_blended_night', 'day_notes',
       'night_notes', 'density_day', 'density_night', 'light_score',
       'buffer_light', 'density_light', 'buffer_day_final_score',
       'buffer_night_final_score', 'density_day_final_score',
       'density_night_final_score'],
      dtype='object')